In [ ]:
# Reproduction of the base paper's own experiment (Fu et al. 2025, Expert Systems With
# Applications -- a DDoS deep-learning survey). Section 6 runs one original experiment:
# a CNN and an LSTM on an 8-class CICDDoS2019 subset, reporting CNN 99.55% across all
# four metrics and LSTM ~96.5%, after just 1 epoch. We reproduce it exactly, then
# evaluate it under our own random/group splits to test whether that score survives a
# split that does not leak behaviorally-identical flows between train and test.
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

sys.path.append(str(Path("..").resolve()))
from src.training.splits import behavior_keys, random_split, group_split, twin_rate
from src.evaluation.metrics import evaluate, plot_confusion

sns.set_theme(style="whitegrid")
keras.utils.set_random_seed(2026)

In [ ]:
PROCESSED_PATH = Path("../data/processed")
FIGURES_PATH = Path("../reports/figures")
SEED = 2026

# The paper's pie chart (Fig. 5) is exactly the 03-11/*.csv files' Label column, not the
# filenames: NetBIOS/BENIGN/LDAP/Portmap/UDP/Syn/MSSQL at 50,000 each and UDPLag at
# 1,873 -- which is precisely the true UDPLag count we measured in 03-11/UDPLag.parquet
# (that file is mostly mislabeled-by-filename Syn/UDP rows; only 1,873 rows are
# genuinely UDPLag). 01-12 is not used: it would add DNS/NTP/SNMP/SSDP/TFTP, none of
# which appear in the paper's class list.
SOURCE_FILES = sorted((PROCESSED_PATH / "03-11").glob("*.parquet"))
ROWS_PER_CLASS = 50_000
PAPER_CLASSES = ["BENIGN", "LDAP", "MSSQL", "NetBIOS", "Portmap", "Syn", "UDP", "UDPLag"]

candidate_features = json.load(open("../reports/feature_config.json"))["numeric_feature_columns"]
print(f"source files: {[f.name for f in SOURCE_FILES]}")
print(f"candidate features: {len(candidate_features)}")

In [ ]:
# Pool every row across the 7 files by its actual Label (not its filename), then sample
# up to ROWS_PER_CLASS per class -- exactly as Fig. 5 describes. UDPLag has only 1,873
# rows total, so it is kept whole, matching the paper's own exceptional 0.5% slice.
rng = np.random.default_rng(SEED)
pools = {cls: [] for cls in PAPER_CLASSES}

for f in SOURCE_FILES:
    chunk = pd.read_parquet(f, columns=candidate_features + ["Label"])
    for cls in PAPER_CLASSES:
        rows = chunk.loc[chunk["Label"] == cls]
        if len(rows):
            pools[cls].append(rows)

data = pd.concat(
    [
        pd.concat(parts, ignore_index=True).sample(
            n=min(ROWS_PER_CLASS, sum(len(p) for p in parts)), random_state=SEED
        )
        for parts in pools.values()
    ],
    ignore_index=True,
).sample(frac=1, random_state=SEED).reset_index(drop=True)  # shuffle; row order matters for the naive split below

print(f"total rows: {len(data):,}")
data["Label"].value_counts()

In [ ]:
# The paper's "valid split" of 0.2 (Table 5) is almost certainly Keras's own
# `validation_split`, which slices the LAST 20% of the array as given -- no shuffling,
# no stratification. We already shuffled once above (typical practice), so this slice
# is at least class-mixed; the point of this cell is to reproduce that exact mechanism,
# not to second-guess it.
y_raw = data["Label"].to_numpy()
classes = sorted(PAPER_CLASSES)
class_to_idx = {c: i for i, c in enumerate(classes)}
y = np.array([class_to_idx[c] for c in y_raw])

X = StandardScaler().fit_transform(data[candidate_features].fillna(0.0))  # naive fill, matching the paper's lack of stated imputation
X = X.reshape(X.shape[0], X.shape[1], 1)  # (samples, features-as-"timesteps", 1 channel)

n_val = int(0.2 * len(X))
split_paper = np.zeros(len(X), dtype=bool)
split_paper[-n_val:] = True  # True = validation/"test" row, matching Keras's trailing slice

behavior_cols = [c for c in candidate_features if c not in ("Source Port", "Destination Port")]
keys = behavior_keys(data, behavior_cols)
split_random = random_split(y, test_size=0.2, seed=SEED)
split_group = group_split(y, keys, test_size=0.2, seed=SEED)

splits = {"paper (trailing 20%)": split_paper, "random": split_random, "group": split_group}
for name, mask in splits.items():
    print(f"{name:<22} test rows: {mask.sum():>7,}   twin rate: {100*twin_rate(keys, mask):5.1f}%")

In [ ]:
def build_cnn(n_features, n_classes):
    return keras.Sequential([
        layers.Input(shape=(n_features, 1)),
        layers.Conv1D(32, kernel_size=3, activation="relu"),
        layers.MaxPooling1D(2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ])


def build_lstm(n_features, n_classes):
    return keras.Sequential([
        layers.Input(shape=(n_features, 1)),
        layers.LSTM(128, return_sequences=True),
        layers.LSTM(64),
        layers.Dense(128, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ])


def compile_and_fit(model, X_train, y_train, epochs):
    model.compile(optimizer=keras.optimizers.Adam(0.001),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(X_train, y_train, batch_size=64, epochs=epochs, verbose=0)
    return model

In [ ]:
EPOCHS = 1  # matches Table 5 exactly; see the extra-epochs cell below for context

results, predictions = [], {}

for split_name, test_mask in splits.items():
    train_mask = ~test_mask
    for model_name, builder in [("CNN", build_cnn), ("LSTM", build_lstm)]:
        start = time.time()
        model = compile_and_fit(builder(X.shape[1], len(classes)), X[train_mask], y[train_mask], EPOCHS)
        y_pred = model.predict(X[test_mask], verbose=0).argmax(axis=1)
        elapsed = time.time() - start

        summary, _ = evaluate(y[test_mask], y_pred, classes)
        results.append({"split": split_name, "model": model_name, **summary, "seconds": round(elapsed, 1)})
        predictions[(split_name, model_name)] = (y[test_mask], y_pred)
        acc = summary["accuracy"]
        f1 = summary["macro_f1"]
        print(f"[{split_name:<22}] {model_name:<4} accuracy={acc:.4f}  macro-F1={f1:.4f}  ({elapsed:.0f}s)")

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("../reports/base_paper_reproduction.csv", index=False)
results_df.pivot(index="model", columns="split", values=["accuracy", "macro_f1"]).round(4)

In [ ]:
for split_name in splits:
    y_true, y_pred = predictions[(split_name, "CNN")]
    plot_confusion(y_true, y_pred, classes, f"Base-paper CNN (1 epoch) -- {split_name} split",
                   FIGURES_PATH / f"confusion_base_paper_CNN_{split_name.split()[0]}.png")
    plt.show()

In [ ]:
# Context, not part of the reproduction: does the 1-epoch result reflect the
# architecture\'s real ceiling, or just an undertrained model? Same CNN, same data,
# trained properly (20 epochs) under the honest group split.
EXTRA_EPOCHS = 20
model = compile_and_fit(build_cnn(X.shape[1], len(classes)), X[~split_group], y[~split_group], EXTRA_EPOCHS)
y_pred = model.predict(X[split_group], verbose=0).argmax(axis=1)
summary, _ = evaluate(y[split_group], y_pred, classes)

one_epoch_f1 = results_df.loc[(results_df["split"] == "group") & (results_df["model"] == "CNN"), "macro_f1"].iloc[0]
print(f"CNN, group split, {EPOCHS} epoch:  macro-F1 = {one_epoch_f1:.4f}")
print(f"CNN, group split, {EXTRA_EPOCHS} epochs: macro-F1 = {summary['macro_f1']:.4f}")